# Agentic AI - Turning functions into tools

## 1. Introduction

### 1.1. Notebook overview

In this notebook, you will create a set of tools to give to an LLM. You will see how the LLM requests tools to be used and also the LLM choosing certain tools when relevant to its task.

### 1.2 Learning outcome

Apply tool-calling design patterns to agent workflows.

To achieve this you will give LLMs controlled access to python functions via `create_response`, manage parameter passing and execution flow, and validate multi-step outputs generated through tool orchestration.


## 2. Setup: Initialize environment and client

As in previous labs, you will begin by initializing your environment. You will import several packages now and also later on as you build tools for your LLM.

In [ ]:
import json
import display_functions
from gates_openai import create_response

## 3. Build your first tool

### 3.1 Defining your function

Now that you have set up your environment. It is time to create you first tool. Run the cell below to define a function that returns the current time as a string.

In [ ]:
# pip install openai-agents

In [ ]:
from datetime import datetime

def get_current_time():
    """
    Returns the current time as a string.
    """
    return datetime.now().strftime("%H:%M:%S")

Test out your function to what exactly this function returns.

In [ ]:
get_current_time()

Great! Just as expected, the function returns a string that has your current time.

### 3.2 Turning your function into an LLM tool

Now, let's use `create_response` to pass this tool to an LLM and get a response. To set up your tool, you first set up the `response` from the LLM. Creating a response first requires creating the message structure. The message structure includes the prompt the user asks, as well as a dictionary that represents the conversation history and each message having a `role` (e.g., "user", "assistant", "system") and `content`.

In [ ]:
# Message structure
prompt = "What time is it?"
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

tools = [
    {
        "type": "function",
        "name": "get_current_time",
        "description": "Returns the current time as a string.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
        }
    }
]

TOOL_MAPPING = {
    "get_current_time": get_current_time
}

After defining your message structure you can construct your create_response method. This will make the LLM call for you and return the result. Let's take a look at the parameters in this call.
* `model`: The model that will be used
* `input`: The list of messages passed to the LLM
* `tools`: The list of tools that the LLM has access to

Run the cell below to call the LLM and see the response.

In [ ]:
response = create_response(
    model="gpt-4o-mini",
    input=messages,
    tools=tools,
)

# See the LLM response
function_calls = [
    item for item in response.output
    if item.type == "function_call"
]

tool_outputs = []

for call in function_calls:
    args = json.loads(call.arguments)
    name = call.name
    result = TOOL_MAPPING[name](**args)

    tool_output = "\n".join(map(str, result))

    tool_outputs.append({
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": tool_output
    })


response = create_response(
    model = "gpt-4o-mini",
    input = tool_outputs,
    tools = tools,
    previous_response_id = response.id
)

print(response.output_text)


And just like that you've given your LLM access to tools! The tool turned your function into a tool that augmented the LLM's knowledge about the world.

### 3.3 Taking a closer look at the response
While the final response's content was just what was expected, there is actually a lot happening behind the scenes in the `response`. Let's use a helpful `utility` function to take a closer look. `pretty_print_response` will extract the steps from the response and show you the important parts in an easy to read format.

In [ ]:
display_functions.pretty_print_response(response)

As you can see, the LLM sent a message to use `get_current_time`. This was executed on your machine and sent back to the LLM. Finally, the LLM, having the full conversation history, used that information to give the final response. `create_response` handled all the complexities of pulling out the message with the tool call, executing it locally.

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

<strong>Congratulations!</strong>

You’ve completed the notebook on **turning functions into tools**.  
Along the way, <strong>you</strong> exposed a Python function as a tool and let the LLM call them.

With these skills, <strong>you</strong> can design agentic workflows that combine LLM reasoning with real actions—reliable, auditable, and easy to extend. 

</div>
